In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re

from pypdf import PdfReader

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")
SOLICITATION_ID = "commerce-ga-25-012"

manifest_folder = PROJECT / "data/processed/manifests"

candidates = sorted(
    manifest_folder.glob(f"{SOLICITATION_ID}_*.json"),
    reverse=True,
)

manifest = None

for path in candidates:
    candidate = json.loads(path.read_text(encoding="utf-8"))

    if candidate.get("packet_download_complete"):
        manifest = candidate
        manifest_path = path
        break

if manifest is None:
    raise RuntimeError("No complete ingestion manifest found.")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = PROJECT / "data/processed/text" / SOLICITATION_ID / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)

print("Manifest:", manifest_path)
print("Documents:", len(manifest["records"]))
print("Extraction output:", OUTPUT)

Manifest: /home/jovyan/project/georgia-bid-readiness/data/processed/manifests/commerce-ga-25-012_20261016T223117221997Z.json
Documents: 12
Extraction output: /home/jovyan/project/georgia-bid-readiness/data/processed/text/commerce-ga-25-012/20261016T225302199674Z


In [2]:
import importlib.metadata as metadata

page_records = []
document_summary = []

for document in manifest["records"]:
    if document["status"] != "ok":
        continue

    pdf_path = PROJECT / document["local_path"]
    document_id = document["document_id"]
    document_pages = []

    try:
        content = pdf_path.read_bytes()
        actual_hash = hashlib.sha256(content).hexdigest()

        if actual_hash != document["sha256"]:
            raise ValueError("PDF hash differs from the ingestion manifest.")

        reader = PdfReader(pdf_path)

        if len(reader.pages) != document["page_count"]:
            raise ValueError("Page count differs from the ingestion manifest.")

        for page_number, page in enumerate(reader.pages, start=1):
            record = {
                "extraction_run_id": RUN_ID,
                "document_id": document_id,
                "solicitation_id": SOLICITATION_ID,
                "document_sha256": actual_hash,
                "source_url": document["source_url"],
                "source_pdf": document["local_path"],
                "page_number": page_number,
                "extraction_method": "pypdf",
                "extractor_version": metadata.version("pypdf"),
            }

            try:
                raw_text = page.extract_text() or ""

                # Light cleanup only; retain the original extraction too.
                clean_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
                clean_text = re.sub(r"[ \t]+", " ", clean_text)
                clean_text = "\n".join(
                    line.strip() for line in clean_text.splitlines()
                ).strip()

                non_whitespace_chars = len(
                    re.sub(r"\s", "", raw_text)
                )

                flags = []
                if not raw_text.strip():
                    flags.append("empty_text")
                elif non_whitespace_chars < 100:
                    flags.append("low_text")

                record.update({
                    "status": "ok",
                    "raw_text": raw_text,
                    "clean_text": clean_text,
                    "character_count": len(raw_text),
                    "non_whitespace_characters": non_whitespace_chars,
                    "quality_flags": flags,
                    "needs_review": bool(flags),
                })

            except Exception as error:
                record.update({
                    "status": "failed",
                    "raw_text": None,
                    "clean_text": None,
                    "character_count": None,
                    "non_whitespace_characters": None,
                    "quality_flags": ["extraction_error"],
                    "needs_review": True,
                    "error": str(error),
                })

            page_records.append(record)
            document_pages.append(record)

        failed_pages = sum(
            p["status"] == "failed" for p in document_pages
        )
        review_pages = sum(p["needs_review"] for p in document_pages)

        document_summary.append({
            "document_id": document_id,
            "status": "partial_failure" if failed_pages else "ok",
            "page_count": len(document_pages),
            "failed_pages": failed_pages,
            "pages_needing_review": review_pages,
        })

        print(
            f"{document_id}: {len(document_pages)} pages, "
            f"{review_pages} flagged for review",
            flush=True,
        )

    except Exception as error:
        document_summary.append({
            "document_id": document_id,
            "status": "failed",
            "error": str(error),
        })
        print(f"FAILED: {document_id}: {error}", flush=True)

pages_path = OUTPUT / "pages.jsonl"

with pages_path.open("w", encoding="utf-8") as file:
    for record in page_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

summary = {
    "extraction_run_id": RUN_ID,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_manifest": manifest_path.relative_to(PROJECT).as_posix(),
    "extractor": "pypdf",
    "extractor_version": metadata.version("pypdf"),
    "low_text_threshold": 100,
    "expected_pages": manifest["total_pages"],
    "page_records_written": len(page_records),
    "successful_pages": sum(p["status"] == "ok" for p in page_records),
    "failed_pages": sum(p["status"] == "failed" for p in page_records),
    "pages_needing_review": sum(p["needs_review"] for p in page_records),
    "documents": document_summary,
}

summary["all_pages_extracted"] = (
    summary["successful_pages"] == summary["expected_pages"]
    and all(d["status"] == "ok" for d in document_summary)
)

(OUTPUT / "extraction_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("\n--- EXTRACTION SUMMARY ---")
for key in [
    "expected_pages",
    "page_records_written",
    "successful_pages",
    "failed_pages",
    "pages_needing_review",
    "all_pages_extracted",
]:
    print(f"{key}: {summary[key]}")

print("Saved:", pages_path)

commerce-ga-25-012-original: 15 pages, 1 flagged for review
commerce-ga-25-012-addendum-01: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-02: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-03: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-04: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-05: 2 pages, 0 flagged for review
commerce-ga-25-012-addendum-06: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-07: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-08: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-09: 1 pages, 0 flagged for review
commerce-ga-25-012-addendum-10: 2 pages, 0 flagged for review
commerce-ga-25-012-addendum-11: 1 pages, 0 flagged for review

--- EXTRACTION SUMMARY ---
expected_pages: 28
page_records_written: 28
successful_pages: 28
failed_pages: 0
pages_needing_review: 1
all_pages_extracted: True
Saved: /home/jovyan/project/georgia-bid-readiness/data/processed/text/commerce-ga-25-012/20261016T2

In [3]:
print("--- FIRST-PAGE PREVIEWS ---")

for page in page_records:
    if page["page_number"] == 1:
        print(f"\n{page['document_id']}")
        print((page["raw_text"] or "[No extracted text]")[:450])

print("\n--- PAGES REQUIRING REVIEW ---")

flagged = [page for page in page_records if page["needs_review"]]

if not flagged:
    print("No pages flagged by the initial screening rules.")
else:
    for page in flagged:
        print(
            page["document_id"],
            "| PDF page", page["page_number"],
            "|", ", ".join(page["quality_flags"]),
        )

--- FIRST-PAGE PREVIEWS ---

commerce-ga-25-012-original
City of Commerce 
P.O. Box 348 
110 State Street 
Commerce, GA 30529 
Phone: (706) 335-3164 
Email: bids@commercega.gov 
 
CITY OF COMMERCE, GEORGIA 
REQUEST FOR PROPOSALS 
FOR 
Janitorial Services for City of Commerce Municipal Buildings 
 
Introduction 
The City of Commerce is seeking proposals from qualified and experienced janitorial service 
providers to provide cleaning services for multiple City buildings. The goal is to maintain a clean 


commerce-ga-25-012-addendum-01
 
ADDENDUM #1 
RFP 25-012 
 
JANITORIAL SERVICE CITY OF COMMERCE 
 
 
This addendum is being issued to include additional qualifications required by Offeror on RFP 25-012: 
 
1. GENERAL SCOPE OF WORK (on page 7 of RFP 25-012) 
We have reviewed the bid documents and are unclear if this proposal requires provision of 
supplies?  City of Commerce, Georgia will provide all supplies and equipment necessary to 
the selected Contractor for the purposes 

commerce

In [5]:
from IPython.display import display, FileLink

flagged_page = next(
    page for page in page_records
    if page["document_id"] == "commerce-ga-25-012-original"
    and page["page_number"] == 10
)

print("PDF page:", flagged_page["page_number"])
print("Character count:", flagged_page["character_count"])
print(
    "Non-whitespace characters:",
    flagged_page["non_whitespace_characters"],
)
print("Flags:", flagged_page["quality_flags"])
print("\n--- EXTRACTED TEXT ---")
print(repr(flagged_page["raw_text"]))

# Create a one-page review copy without modifying the source PDF.
from pypdf import PdfReader, PdfWriter

source = PROJECT / flagged_page["source_pdf"]
review_folder = PROJECT / "reports/page_review"
review_folder.mkdir(parents=True, exist_ok=True)

review_file = review_folder / "commerce_original_page_10.pdf"

reader = PdfReader(source)
writer = PdfWriter()
writer.add_page(reader.pages[9])  # PDF page 10 uses index 9.

with review_file.open("wb") as file:
    writer.write(file)

print("\nReview copy:", review_file)

# Link relative to the notebook's working directory.
import os
display(FileLink(os.path.relpath(review_file, Path.cwd())))

PDF page: 10
Character count: 36
Non-whitespace characters: 26
Flags: ['low_text']

--- EXTRACTED TEXT ---
' \n \n \nRequired Forms for Submission '

Review copy: /home/jovyan/project/georgia-bid-readiness/reports/page_review/commerce_original_page_10.pdf


/home/jovyan/project/georgia-bid-readiness/reports/page_review/commerce_original_page_10.pdf

In [6]:
from datetime import datetime, timezone
import json

expected_heading = "required forms for submission"
extracted = " ".join(
    (flagged_page["raw_text"] or "").lower().split()
)

if expected_heading not in extracted:
    raise ValueError(
        "The heading is not present in the extracted text. "
        "Review the extraction before marking this page resolved."
    )

review = {
    "extraction_run_id": flagged_page["extraction_run_id"],
    "document_id": flagged_page["document_id"],
    "document_sha256": flagged_page["document_sha256"],
    "page_number": flagged_page["page_number"],
    "original_quality_flags": flagged_page["quality_flags"],
    "review_status": "resolved",
    "resolution": "legitimate_low_text_page",
    "ocr_required": False,
    "reviewed_by": "Ibrahim Shittu",
    "reviewed_at_utc": datetime.now(timezone.utc).isoformat(),
    "review_method": "visual inspection of PDF page",
    "notes": (
        "Page contains only the heading "
        "'Required Forms for Submission'. "
        "The heading is present in the extracted text."
    ),
}

review_path = OUTPUT / "page_reviews.jsonl"

with review_path.open("a", encoding="utf-8") as file:
    file.write(json.dumps(review, ensure_ascii=False) + "\n")

print("Review saved:", review_path)
print("Page 10: resolved; OCR not required.")

Review saved: /home/jovyan/project/georgia-bid-readiness/data/processed/text/commerce-ga-25-012/20261016T225302199674Z/page_reviews.jsonl
Page 10: resolved; OCR not required.
